<a href="https://colab.research.google.com/github/JhonPiterson/TCC-Previs-o-de-Dificuldade-Financeira-com-XGBoost/blob/main/TCC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pandas openpyxl xgboost scikit-learn matplotlib seaborn

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_curve, auc, brier_score_loss, classification_report
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
import joblib
from imblearn.under_sampling import RandomUnderSampler
from scipy.stats import ttest_ind

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
caminho_dados = ""
caminho_resultados = ""

In [ ]:
data0 = pd.read_excel(caminho_dados, sheet_name="final")

In [ ]:
data = data0.loc[data0.iloc[:, 4] == 'Utilities']

In [ ]:
vars = list(range(0, 5)) + [6, 7, 9, 10, 13, 14, 16, 17, 18, 19, 20, 21, 22, 23, 25]
t1 = list(range(26, 30))
t2 = list(range(30, 34))
t3 = list(range(34, 38))

In [ ]:
data1 = data.iloc[:, [i for i in vars + t1 if i < data.shape[1]]]
data2 = data.iloc[:, [i for i in vars + t2 if i < data.shape[1]]]
data3 = data.iloc[:, [i for i in vars + t3 if i < data.shape[1]]]

In [ ]:
column_names = ["ID", "YEAR", "Name", "Country", "Sector", "LIQ", "LVC",
                "NIE", "NICA", "EPS", "DTE", "X1A", "X2A", "X3A", "X4A", "X5A",
                "OPM", "CROE", "CPB", "GSA", "FD1", "FD2", "FD3", "FD4"]
data1.columns = column_names[:len(data1.columns)]
data2.columns = column_names[:len(data2.columns)]
data3.columns = column_names[:len(data3.columns)]

In [ ]:
dataset11 = data1[data1["FD1"].notna()].copy()
dataset11 = dataset11.drop(columns=["FD2", "FD3", "FD4"])
dataset11.rename(columns={"FD1": "FD"}, inplace=True)

dataset12 = data2[data2["FD1"].notna()].copy()
dataset12 = dataset12.drop(columns=["FD2", "FD3", "FD4"])
dataset12.rename(columns={"FD1": "FD"}, inplace=True)

dataset13 = data3[data3["FD1"].notna()].copy()
dataset13 = dataset13.drop(columns=["FD2", "FD3", "FD4"])
dataset13.rename(columns={"FD1": "FD"}, inplace=True)

dataset21 = data1[data1["FD2"].notna()].copy()
dataset21 = dataset21.drop(columns=["FD1", "FD3", "FD4"])
dataset21.rename(columns={"FD2": "FD"}, inplace=True)

dataset22 = data2[data2["FD2"].notna()].copy()
dataset22 = dataset22.drop(columns=["FD1", "FD3", "FD4"])
dataset22.rename(columns={"FD2": "FD"}, inplace=True)

dataset23 = data3[data3["FD2"].notna()].copy()
dataset23 = dataset23.drop(columns=["FD1", "FD3", "FD4"])
dataset23.rename(columns={"FD2": "FD"}, inplace=True)

dataset31 = data1[data1["FD3"].notna()].copy()
dataset31 = dataset31.drop(columns=["FD1", "FD2", "FD4"])
dataset31.rename(columns={"FD3": "FD"}, inplace=True)

dataset32 = data2[data2["FD3"].notna()].copy()
dataset32 = dataset32.drop(columns=["FD1", "FD2", "FD4"])
dataset32.rename(columns={"FD3": "FD"}, inplace=True)

dataset33 = data3[data3["FD3"].notna()].copy()
dataset33 = dataset33.drop(columns=["FD1", "FD2", "FD4"])
dataset33.rename(columns={"FD3": "FD"}, inplace=True)

dataset41 = data1[data1["FD4"].notna()].copy()
dataset41 = dataset41.drop(columns=["FD1", "FD2", "FD3"])
dataset41.rename(columns={"FD4": "FD"}, inplace=True)

dataset42 = data2[data2["FD4"].notna()].copy()
dataset42 = dataset42.drop(columns=["FD1", "FD2", "FD3"])
dataset42.rename(columns={"FD4": "FD"}, inplace=True)

dataset43 = data3[data3["FD4"].notna()].copy()
dataset43 = dataset43.drop(columns=["FD1", "FD2", "FD3"])
dataset43.rename(columns={"FD4": "FD"}, inplace=True)

In [ ]:
# Lista dos datasets filtrados por setor Utilities
datasets = {
    'dataset11': dataset11, 'dataset12': dataset12, 'dataset13': dataset13,
    'dataset21': dataset21, 'dataset22': dataset22, 'dataset23': dataset23,
    'dataset31': dataset31, 'dataset32': dataset32, 'dataset33': dataset33,
    'dataset41': dataset41, 'dataset42': dataset42, 'dataset43': dataset43,
}

encoded_datasets = {}

for name, df in datasets.items():
    df_copy = df.copy()

    # Elimina a coluna 'Sector' pois o setor é fixo (apenas Utilities)
    df_copy = df_copy.drop(columns=["Sector"])

    # Dummifica apenas 'Country'
    cat_vars = df_copy[["Country"]]
    other_vars = df_copy.drop(columns=["Country", "FD"])
    y = df_copy["FD"]

    # Aplica OneHotEncoder com drop='first' para evitar multicolinearidade
    encoder = OneHotEncoder(drop="first", sparse_output=False)
    encoded = encoder.fit_transform(cat_vars)
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(["Country"]))

    # Junta dados numéricos + dummies + variável alvo
    final_df = pd.concat([other_vars.reset_index(drop=True),
                          encoded_df.reset_index(drop=True),
                          y.reset_index(drop=True)], axis=1)

    encoded_datasets[name] = final_df


In [ ]:
def corrigir_predicoes(test, X_test, y_test, model):
    X_test_clean = X_test.dropna()
    y_test_clean = y_test.loc[X_test_clean.index]
    y_proba = model.predict_proba(X_test_clean)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    test_full = test.copy()
    test_full["XGB"] = np.nan
    test_full["XGB-prob"] = np.nan
    test_full.loc[X_test_clean.index, "XGB"] = y_pred
    test_full.loc[X_test_clean.index, "XGB-prob"] = np.round(y_proba, 4)

    return test_full, y_pred, y_proba, y_test_clean

results_path = caminho_resultados
os.makedirs(results_path, exist_ok=True)

for name, df in encoded_datasets.items():
    k = 2015
    train = df[df['YEAR'] < k]
    test = df[df['YEAR'] >= k]

    train_input_cols = [c for c in train.columns if c not in ["ID", "YEAR", "Name"]]
    test_input_cols = [c for c in test.columns if c not in ["ID", "YEAR", "Name"]]

    train[train_input_cols] = train[train_input_cols].apply(pd.to_numeric, errors='coerce')
    test[test_input_cols] = test[test_input_cols].apply(pd.to_numeric, errors='coerce')

    X_train = train.drop(columns=["FD", "ID", "YEAR", "Name"])
    y_train = train["FD"].astype(int)
    X_test = test.drop(columns=["FD", "ID", "YEAR", "Name"])
    y_test = test["FD"].astype(int)

    # Undersampling com RandomUnderSampler
    rus = RandomUnderSampler(sampling_strategy='auto', random_state=123)
    X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

    model = XGBClassifier(eval_metric='logloss', random_state=123)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    cv_acc = cross_val_score(model, X_train_bal, y_train_bal, cv=cv, scoring="accuracy")
    cv_auc = cross_val_score(model, X_train_bal, y_train_bal, cv=cv, scoring="roc_auc")

    print(f"{name}: acc={cv_acc.mean():.4f}, auc={cv_auc.mean():.4f}")

    model.fit(X_train_bal, y_train_bal)
    joblib.dump(model, os.path.join(results_path, f"{name}.joblib"))

    importance_df = pd.DataFrame({
        'Platform': X_train_bal.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)
    importance_df.to_csv(os.path.join(results_path, f"varimp-xgb-{name}.csv"), index=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Importance', y='Platform', data=importance_df, color='gray')
    plt.title("XGBoost Feature Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(results_path, f"varimp-xgb-{name}.pdf"))
    plt.close()

    test_full, y_pred, y_proba, y_test_clean = corrigir_predicoes(test, X_test, y_test, model)

    cm = confusion_matrix(y_test_clean, y_pred)
    acc = accuracy_score(y_test_clean, y_pred)
    TN, FP, FN, TP = cm.ravel()

    TError_I = round(100 * FP / (FP + TN), 2) if (FP + TN) > 0 else np.nan
    TError_II = round(100 * FN / (FN + TP), 2) if (FN + TP) > 0 else np.nan

    fpr, tpr, _ = roc_curve(y_test_clean, y_proba)
    roc_auc_score = auc(fpr, tpr)
    brier = brier_score_loss(y_test_clean, y_proba)
    ks = max(tpr - fpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f'XGBoost (AUC = {roc_auc_score:.2f})', color='blue')
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
    plt.title("ROC Curve")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(results_path, f"ROC-xgb-{name}.png"), dpi=160)
    plt.savefig(os.path.join(results_path, f"ROC-xgb-{name}.pdf"))
    plt.close()

    results_df = pd.DataFrame({
        "ModelxMeasure": ["XGB"],
        "TP": [TP], "TN": [TN], "FP": [FP], "FN": [FN],
        "TError_I": [TError_I], "TError_II": [TError_II],
        "ACC": [acc],
        "SENS": [TP / (TP + FN) if TP + FN > 0 else np.nan],
        "SPEC": [TN / (TN + FP) if TN + FP > 0 else np.nan],
        "BS": [brier],
        "AUC": [roc_auc_score],
        "KS": [ks]
    })

    with pd.ExcelWriter(os.path.join(results_path, f"Performance-{name}.xlsx")) as writer:
        results_df.to_excel(writer, sheet_name="metrics", index=False)
        train.to_excel(writer, sheet_name="train", index=False)
        test_full.to_excel(writer, sheet_name="testfull", index=False)
